# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and analyzing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant Schema URL:**

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect all available record sets in the dataset (accessible by their `@id`), along with their fields and column `@id`s.

In [ ]:
# Print available record sets, fields, and columns (@id only) for reference
for recset in metadata.record_sets:
    print(f"RecordSet @id: {recset.id}")
    print(f"  Name: {recset.name}")
    print(f"  Description: {getattr(recset, 'description', '')}")
    if hasattr(recset, 'fields'):
        for field in recset.fields:
            print(f"    Field @id: {field.id}, name: {getattr(field, 'name', '')}")
            if hasattr(field, 'column') and field.column:
                if isinstance(field.column, list):
                    for col in field.column:
                        print(f"      Column @id: {col.id}, name: {getattr(col, 'name', '')}")
                else:
                    print(f"      Column @id: {field.column.id}, name: {getattr(field.column, 'name', '')}")
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this notebook, we will work with the main protein expression record set. For this dataset, the key quantitative information resides in the main table (record set).

**Note:** For your exploration, update the `record_sets_to_load` list below with the exact `@id`(s) you want to analyze, as printed in the previous cell.

In [ ]:
# Example: Specify the RecordSet @id(s) you'd like to extract (update as appropriate based on overview above):
# For this dataset, the main table appears to use '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/recordset/protein_table' (see printed record sets above)

record_sets_to_load = [
    'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/recordset/protein_table'
]

dataframes = {}
for recset_id in record_sets_to_load:
    print(f"Extracting records for RecordSet @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded columns: {dataframes[recset_id].columns.tolist()}")
    display(dataframes[recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the proteins table to filter, normalize, and explore groupings.

We'll operate using column `@id`s as mapped in the previous steps.

- We'll use total peptide count (or similar quantitative column, update the field as needed based on your extracted DataFrame columns) as our primary measure.
- We'll demonstrate normalization and grouping operations by an "experiment" or "sample" column, if available.

In [ ]:
# Choose the record set to work with
record_set_id = record_sets_to_load[0]
df = dataframes[record_set_id]
# ---
# Identify numeric and group fields by their Croissant @id (as found in overview, e.g.,
#   peptide count: '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/field/total_peptide_count'
#
# For demonstration, use a 'total peptide count' field and 'experiment' or 'sample' as group field
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/field/total_peptide_count'
group_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/field/sample'

# Make sure the field exists (the DataFrame columns should use Field @id as key)
print('DataFrame columns:', df.columns.tolist())
if numeric_field_id not in df.columns:
    raise ValueError(f"Field {numeric_field_id} not found in columns. Available: {df.columns.tolist()}")

threshold = 10

# Filter for rows where peptide count > threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by another field (e.g., sample or experiment), if available
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships. We'll plot the distribution of peptide counts and a boxplot by sample if the grouping field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field (peptide count)
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title('Distribution of Total Peptide Count')
plt.xlabel('Total Peptide Count (@id: ' + numeric_field_id + ')')
plt.ylabel('Frequency')
plt.show()

# Boxplot by group field
if group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title('Peptide count by Sample')
    plt.xlabel('Sample (@id: ' + group_field_id + ')')
    plt.ylabel('Total Peptide Count')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We demonstrated loading and exploring the Mass Spectrometry Analysis dataset via its Croissant schema using `mlcroissant`. 
- We referenced all data structures and columns by their `@id` per best FAIR data practices.
- Data were filtered, normalized, and grouped by key fields, and quantitative summaries and visualizations were produced.

You can further extend this notebook for more advanced analysis, such as statistical modeling or machine learning, using the standardized, semantically-rich field identifiers provided by Croissant.